# Sentinel-2 Processing Workflow

This notebook demonstrates the complete workflow for downloading and processing Sentinel-2 satellite imagery using the `disasters-product-algorithms` package.

## Workflow Steps
1. **Configure Environment Variables** - Set processing parameters
2. **Download Sentinel-2 Data** - Download imagery from Copernicus
3. **Process Sentinel-2 Data** - Generate products with COG conversion and event naming
4. **View Results** - Examine the generated outputs

## Features Demonstrated
- Cloud Optimized GeoTIFF (COG) conversion
- Event-based file naming for disaster response
- Multiple product generation (true color, NDVI, NDWI, MNDWI, NBR, water extent)
- Cloud masking (L2A only)

## 1. Environment Setup

Configure all processing parameters as environment variables for easy modification.

In [ ]:
import os
import subprocess
from pathlib import Path

# ==================== CONFIGURATION ====================
# Modify these variables according to your requirements

# Output directory for downloaded and processed data
OUTPUT_DIR = os.path.expanduser("~/disasters_data/sentinel2")

# Download parameters
TILE_ID = "T17RLN"              # Sentinel-2 tile ID (e.g., T17RLN, T36SYD)
DOWNLOAD_DATE = "20231115"      # Date to download (YYYYMMDD format)
PROCESSING_LEVEL = "2"          # 1 = L1C, 2 = L2A (L2A recommended for cloud masking)

# Processing parameters
PRODUCTS = ["true", "ndvi", "ndwi", "mndwi", "nbr"]  # Products to generate
EVENT_NAME = "202311_Example_Event"                   # Event prefix for output files
PROCESS_DATE = "20231115"                             # Specific date to process (optional)

# COG options
COMPRESSION = "ZSTD"            # Compression type: ZSTD, DEFLATE, LZW
COMPRESSION_LEVEL = 22          # Compression level (22 for ZSTD, 9 for DEFLATE)
NODATA = None                   # No-data value (None = auto-detect)

# Processing flags
ENABLE_MERGE = True             # Merge tiles by date and product
ENABLE_MASK = True              # Apply cloud masking (L2A only)
FORCE_OVERWRITE = False         # Force reprocessing of existing files

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration:")
print(f"  Output Directory: {OUTPUT_DIR}")
print(f"  Tile ID: {TILE_ID}")
print(f"  Download Date: {DOWNLOAD_DATE}")
print(f"  Processing Level: L{PROCESSING_LEVEL}A")
print(f"  Products: {', '.join(PRODUCTS)}")
print(f"  Event Name: {EVENT_NAME}")
print(f"  COG Compression: {COMPRESSION} (level {COMPRESSION_LEVEL})")

## 2. Download Sentinel-2 Data

Download Sentinel-2 imagery from the Copernicus Data Space Ecosystem.

**Note:** You will be prompted for your Copernicus credentials.
- Register at: https://dataspace.copernicus.eu/

In [ ]:
# Build download command
download_cmd = [
    "download_sentinel2",
    OUTPUT_DIR,
    "-tile", TILE_ID,
    "-date", DOWNLOAD_DATE,
    "-level", PROCESSING_LEVEL
]

print("Downloading Sentinel-2 data...")
print(f"Command: {' '.join(download_cmd)}")
print("\nYou will be prompted for Copernicus credentials.\n")

# Execute download
result = subprocess.run(download_cmd, cwd=os.getcwd())

if result.returncode == 0:
    print("\n✓ Download completed successfully!")
else:
    print(f"\n✗ Download failed with return code {result.returncode}")

## 3. Process Sentinel-2 Data

Process the downloaded imagery to generate various products with COG conversion and event naming.

In [ ]:
# Build processing command
process_cmd = [
    "process_sentinel2",
    OUTPUT_DIR,
    "-p", *PRODUCTS,
    "-event", EVENT_NAME,
    "-compression", COMPRESSION,
    "-compression_level", str(COMPRESSION_LEVEL)
]

# Add optional parameters
if PROCESS_DATE:
    process_cmd.extend(["-date", PROCESS_DATE])

if ENABLE_MERGE:
    process_cmd.append("-merge")

if ENABLE_MASK:
    process_cmd.append("-mask")

if FORCE_OVERWRITE:
    process_cmd.append("-force")

if NODATA is not None:
    process_cmd.extend(["-nodata", str(NODATA)])

print("Processing Sentinel-2 data...")
print(f"Command: {' '.join(process_cmd)}")
print()

# Execute processing
result = subprocess.run(process_cmd, cwd=os.getcwd())

if result.returncode == 0:
    print("\n✓ Processing completed successfully!")
else:
    print(f"\n✗ Processing failed with return code {result.returncode}")

## 4. View Results

Examine the generated output files and directory structure.

In [ ]:
import glob

# Find output directory
output_dir = os.path.join(OUTPUT_DIR, "output")

if os.path.exists(output_dir):
    print(f"Output directory: {output_dir}\n")
    
    # List all generated products
    print("Generated Products:")
    print("=" * 80)
    
    # Find all GeoTIFF files
    tif_files = sorted(glob.glob(os.path.join(output_dir, "**/*.tif"), recursive=True))
    
    if tif_files:
        # Group by product type
        product_types = {}
        for tif_file in tif_files:
            # Extract product type from path
            parts = tif_file.split(os.sep)
            if len(parts) >= 3:
                product_type = parts[-2]  # Directory name (e.g., trueColor, NDVI)
                if product_type not in product_types:
                    product_types[product_type] = []
                product_types[product_type].append(os.path.basename(tif_file))
        
        # Display grouped results
        for product_type, files in sorted(product_types.items()):
            print(f"\n{product_type}:")
            for f in files:
                file_size = os.path.getsize(os.path.join(output_dir, product_type, f)) / (1024*1024)
                print(f"  - {f} ({file_size:.1f} MB)")
        
        print(f"\nTotal files: {len(tif_files)}")
    else:
        print("No GeoTIFF files found.")
else:
    print(f"Output directory not found: {output_dir}")

## Output File Naming Convention

Files are named with the event prefix and formatted date:

**Original:** `S2B_MSIL2A_colorInfrared_20251111_161419_T17RLN.tif`

**With Event Naming:** `202311_Example_Event_S2B_MSIL2A_colorInfrared_161419_T17RLN_2025-11-11_day.tif`

This naming convention:
- Adds event prefix for organization
- Removes date from middle position
- Adds formatted date (YYYY-MM-DD) at the end
- Includes `_day` suffix for AWS/cloud compatibility

## Next Steps

You can now:
1. Load and visualize the GeoTIFF files using libraries like `rasterio` or `GDAL`
2. Upload the COG files to cloud storage (S3, GCS, etc.)
3. Process additional dates or tiles by modifying the configuration variables
4. Generate additional products by updating the `PRODUCTS` list